In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from omegaconf import OmegaConf

In [ ]:
CONFIG_PATH = Path("../conf/config.yaml")

In [ ]:
config = OmegaConf.load(CONFIG_PATH)

In [ ]:
from conv_wm.config import get_path

In [ ]:
get_path(config, "ego4d", "raw", "av_train")

In [ ]:
video_info = pd.read_csv(get_path(config, "egocom", "raw", "video_info"))
ground_truth = pd.read_csv(get_path(config, "egocom", "raw", "ground_truth"))
av_train = pd.read_json(get_path(config, "ego4d", "raw", "av_train"))
av_val = pd.read_json(get_path(config, "ego4d", "raw", "av_val"))

In [ ]:
video_info.head()

In [ ]:
ground_truth.head()

In [ ]:
av_train.keys()

In [ ]:
for k, v in av_train.items():
    if isinstance(v, dict):
        print(k, "\n", v.keys())
    else:
        print(k, "\n", v)

In [ ]:
av_train.map(type).nunique()
av_train.dtypes[av_train.dtypes == "object"]

In [ ]:
av_val.keys()

In [ ]:
for k, v in av_val.items():
    if isinstance(v, dict):
        print(k, "\n", v.keys())
    else:
        print(k, "\n", v)

In [ ]:
video_info.to_parquet(get_path(config, "egocom", "interim", "video_info"), index=False)
ground_truth.to_parquet(
    get_path(config, "egocom", "interim", "ground_truth"), index=False
)
av_train.to_parquet(get_path(config, "ego4d", "interim", "av_train"), index=False)
av_val.to_parquet(get_path(config, "ego4d", "interim", "av_val"), index=False)

In [ ]:
def to_interim(cfg, df, dataset, output_key, *, source_keys=None, source_stage="raw"):
    if source_stage not in {"raw", "interim"}:
        raise ValueError("source_stage must be 'raw' or 'interim'")

    if source_keys is None:
        source_keys = [output_key]
    elif isinstance(source_keys, str):
        source_keys = [source_keys]

    source_paths = [
        get_path(cfg, dataset, source_stage, source_key) for source_key in source_keys
    ]
    dst = get_path(cfg, dataset, "interim", output_key)
    dst.parent.mkdir(parents=True, exist_ok=True)

    table = pa.Table.from_pandas(df, preserve_index=False)
    meta = {
        **(table.schema.metadata or {}),
        b"stage": b"interim",
        b"dataset": dataset.encode(),
        b"source_stage": source_stage.encode(),
        b"source_keys": ",".join(source_keys).encode(),
        b"source_paths": "\n".join(map(str, source_paths)).encode(),
    }
    pq.write_table(table.replace_schema_metadata(meta), dst, compression="zstd")
    return dst

In [ ]:
for ds, key, df in [
    ("egocom", "video_info", video_info),
    ("egocom", "ground_truth", ground_truth),
    ("ego4d", "av_train", av_train),
    ("ego4d", "av_val", av_val),
]:
    to_interim(config, df, ds, key)

In [ ]:
video_info = pd.read_parquet(get_path(config, "egocom", "interim", "video_info"))
ground_truth = pd.read_parquet(get_path(config, "egocom", "interim", "ground_truth"))
av = pd.read_parquet(get_path(config, "ego4d", "interim", "av"))

In [ ]:
av = pd.read_parquet(get_path(config, "ego4d", "interim", "av"))

In [ ]:
video_info.head(10)

In [ ]:
video_info.isna().sum()

In [ ]:
video_info.isnull().sum()

In [ ]:
(video_info[["train", "val", "test"]].sum(axis=1) == 1).all()

In [ ]:
video_info.keys()

In [ ]:
video_info.tokenized_words.head()

In [ ]:
video_info.isna().sum()

In [ ]:
video_info.describe()

In [ ]:
video_info.info()

In [ ]:
# (video_info
#     .groupby(["conversation_id", "video_speaker_id"])["word_count"].sum()
#     .unstack("video_speaker_id")
#     .plot(kind="bar", stacked=True, figsize=(12, 5))
# )
# plt.ylabel("word count")
# plt.xlabel("conversation")

In [ ]:
len(video_info["video_id"].unique())

In [ ]:
len(video_info["video_speaker_id"].unique()), video_info["video_speaker_id"].unique()

In [ ]:
len(video_info["num_speakers"].unique()), video_info["num_speakers"].unique()

In [ ]:
(
    len(video_info[video_info["num_speakers"] == 3]),
    len(video_info[video_info["num_speakers"] == 2]),
)

In [ ]:
len(video_info["conversation_id"].unique())

In [ ]:
len(video_info["cid"].unique())

In [ ]:
video_info[["conversation_id", "cid"]].head()

In [ ]:
video_info[["conversation_id", "video_speaker_id"]].head(10)

In [ ]:
len(video_info[video_info.background_fan])

In [ ]:
len(video_info[video_info.background_music])

In [ ]:
len(video_info[~video_info.native_speaker])

In [ ]:
len(video_info[video_info.train])

In [ ]:
len(video_info[video_info.val])

In [ ]:
len(video_info[video_info.test])

In [ ]:
ground_truth.head()

In [ ]:
ground_truth.columns

In [ ]:
ground_truth.describe()

In [ ]:
ground_truth.info()

In [ ]:
ground_truth.isna().sum()

In [ ]:
ground_truth.isnull().sum()

In [ ]:
len(ground_truth[ground_truth["startTime"] == 0.0])

In [ ]:
ground_truth.groupby(["conversation_id", "speaker_id"])["startTime"].min()

In [ ]:
len(ground_truth[ground_truth["endTime"].isna() & ground_truth["startTime"].isna()])

In [ ]:
set(
    ground_truth[ground_truth["endTime"].isna() & ground_truth["startTime"].isna()].word
)

In [ ]:
na_ws = ground_truth[
    (ground_truth["endTime"].isna())
    & (ground_truth["startTime"].isna())
    & (ground_truth["word"] == " ")
]

In [ ]:
len(na_ws)

In [ ]:
na_nws = ground_truth[
    (ground_truth["endTime"].isna())
    & (ground_truth["startTime"].isna())
    & (ground_truth["word"] != " ")
]

In [ ]:
len(na_nws)

In [ ]:
set(na_nws[na_nws.conversation_id == "day_1__con_1__part2"].word)

In [ ]:
ground_truth_na = ground_truth.copy().dropna().reset_index(drop=True)

In [ ]:
ground_truth_es_na = ground_truth[
    ground_truth["endTime"].isna() & ground_truth["startTime"].isna()
].copy()

In [ ]:
ground_truth_es_na.info()

In [ ]:
len(ground_truth_na[ground_truth_na.word == " "])

In [ ]:
len(ground_truth_es_na[ground_truth_es_na.word == " "])

In [ ]:
ground_truth_na.info()

In [ ]:
ground_truth_na["word"].value_counts()

In [ ]:
dup = ground_truth_na[
    ground_truth_na.duplicated(subset=["startTime", "endTime"], keep=False)
]
dup.sort_values(["startTime", "endTime"])

In [ ]:
ground_truth_na_merged = ground_truth_na.groupby(
    ["conversation_id", "speaker_id", "startTime", "endTime"],
    as_index=False,
    sort=False,
).agg(
    word=("word", "".join)  # ← séparateur, mets "" si tu veux collé
)
ground_truth_na_merged

In [ ]:
ground_truth_na_merged[
    ground_truth_na_merged["word"].str.fullmatch(r"\d+", na=False)
].copy()

In [ ]:
ground_truth_es_na["word"].value_counts()

In [ ]:
ground_truth_es_na_w = ground_truth_es_na[
    ground_truth_es_na["word"].str.contains(r"\w", na=False)
].copy()

In [ ]:
len(ground_truth_es_na), len(ground_truth_es_na_w)

In [ ]:
ground_truth_e_na = ground_truth[
    ground_truth["endTime"].isna() & ground_truth["startTime"].notna()
].copy()

In [ ]:
ground_truth_e_na.word.unique()

In [ ]:
ground_truth_w_na = ground_truth[
    ground_truth["endTime"].notna()
    & ground_truth["startTime"].notna()
    & ground_truth["word"].isna()
].copy()

In [ ]:
ground_truth_w_na

In [ ]:
set(
    ground_truth[
        ground_truth["endTime"].isna() & ground_truth["startTime"].isna()
    ].conversation_id
)

In [ ]:
ground_truth[ground_truth["endTime"].isna() & ground_truth["startTime"].notna()]

In [ ]:
mask = ground_truth["endTime"].isna() & ground_truth["startTime"].notna()

In [ ]:
ground_truth.loc[mask, "word"].value_counts().head(30)

In [ ]:
ground_truth[ground_truth["word"].isna()]

In [ ]:
ground_truth.word[2] == " "

In [ ]:
ground_truth[ground_truth["endTime"].isna() & ground_truth["startTime"].isna()].head(20)

In [ ]:
ground_truth["conversation_id"].unique()

In [ ]:
video_info["conversation_id"].unique()

In [ ]:
video_info

In [ ]:
len(video_info.groupby(["conversation_id", "video_speaker_id"]))

In [ ]:
len(ground_truth.groupby(["conversation_id", "speaker_id"]))

In [ ]:
ground_truth.assign(
    duration=ground_truth["endTime"] - ground_truth["startTime"]
).groupby(["conversation_id", "speaker_id"])["duration"].sum()

In [ ]:
# speaker_shares = (
#     ground_truth
#         .assign(duration=ground_truth["endTime"] - ground_truth["startTime"])
#         .groupby(["conversation_id", "speaker_id"])["duration"].sum()
#         .to_frame("speak_time")
#         .join(
#             video_info.groupby(["conversation_id", "video_speaker_id"])["word_count"]
#                       .sum().rename("words")
#         )
#         .dropna()
# )

# speaker_shares["time_share"] = (
#     speaker_shares["speak_time"] /
#     speaker_shares.groupby("conversation_id")["speak_time"].transform("sum")
# )
# speaker_shares["word_share"] = (
#     speaker_shares["words"] /
#     speaker_shares.groupby("conversation_id")["words"].transform("sum")
# )
# speaker_shares
# fig, ax = plt.subplots(figsize=(7, 7))
# ax.scatter(speaker_shares["time_share"], speaker_shares["word_share"], alpha=0.6, s=30)
# ax.plot([0, 1], [0, 1], ls="--", color="gray", lw=1)
# ax.set_xlabel("part du temps de parole dans la conversation")
# ax.set_ylabel("part des mots dans la conversation")
# ax.set_xlim(0, 1); ax.set_ylim(0, 1)
# ax.set_aspect("equal")
# plt.show()


In [ ]:
# import numpy as np

# # 1. Débit (mots/seconde) par speaker
# wps = (speaker_shares
#        .assign(wps=speaker_shares["words"] / speaker_shares["speak_time"])
#        .reset_index())
# wps = wps[np.isfinite(wps["wps"])]   # supprime les inf (speak_time == 0)
# speaker_col = [c for c in ("video_speaker_id", "speaker_id") if c in wps.columns][0]

# fig, ax = plt.subplots(figsize=(8, 4))
# wps.boxplot(column="wps", by=speaker_col, ax=ax)

# ax.set_title("Débit par speaker")
# ax.set_xlabel(speaker_col); ax.set_ylabel("mots / seconde")
# plt.suptitle("")
# plt.show()


In [ ]:
# meta = video_info.groupby(["conversation_id", "video_speaker_id"])[
#     ["native_speaker", "speaker_gender"]
# ].first()
# wps_meta = wps.set_index(["conversation_id", speaker_col]).join(meta).reset_index()

# fig, ax = plt.subplots(figsize=(8, 4))
# bins = np.linspace(wps_meta["wps"].min(), wps_meta["wps"].max(), 26)
# groups = [g["wps"].values for _, g in wps_meta.groupby("native_speaker")]
# labels = [f"native={k} (n={len(g)})" for k, g in wps_meta.groupby("native_speaker")]
# weights = [np.ones(len(g)) / len(g) for g in groups]
# ax.hist(groups, bins=bins, weights=weights, stacked=True, label=labels, edgecolor="white")
# ax.set_xlabel("mots / seconde"); ax.set_ylabel("proportion du groupe"); ax.legend()
# plt.show()


In [ ]:
# fig, ax = plt.subplots(figsize=(8, 4))
# bins = np.linspace(wps_meta["wps"].min(), wps_meta["wps"].max(), 26)
# groups = [g["wps"].values for _, g in wps_meta.groupby("speaker_gender")]
# labels = [f"{k} (n={len(g)})" for k, g in wps_meta.groupby("speaker_gender")]
# weights = [np.ones(len(g)) / len(g) for g in groups]
# ax.hist(groups, bins=bins, weights=weights, stacked=True, label=labels, edgecolor="white")
# ax.set_xlabel("mots / seconde"); ax.set_ylabel("proportion du groupe"); ax.legend()
# plt.show()


In [ ]:
# fig, ax = plt.subplots(figsize=(8, 4))
# bins = np.linspace(wps["wps"].min(), wps["wps"].max(), 26)
# groups = [g["wps"].values for _, g in wps.groupby(speaker_col)]
# labels = [f"speaker {k} (n={len(g)})" for k, g in wps.groupby(speaker_col)]
# weights = [np.ones(len(g)) / len(g) for g in groups]
# ax.hist(groups, bins=bins, weights=weights, stacked=True, label=labels, edgecolor="white")
# ax.set_xlabel("mots / seconde"); ax.set_ylabel("proportion du groupe")
# ax.legend(title=speaker_col)
# plt.show()


In [ ]:
# conv = "day_1__con_1__part1"
# sub = ground_truth[ground_truth.conversation_id == conv].dropna(subset=["startTime", "endTime"])

# fig, ax = plt.subplots(figsize=(14, 3))
# for spk, g in sub.groupby("speaker_id"):
#     ax.hlines([spk] * len(g), g["startTime"].values, g["endTime"].values, lw=6)
# ax.set_yticks(sorted(sub.speaker_id.unique()))
# ax.set_xlabel("temps (s)"); ax.set_ylabel("speaker")
# ax.set_title(f"Tours de parole — {conv}")
# plt.show()


In [ ]:
# # 3. Sanity-check des splits train/val/test vs caractéristiques
# split = video_info[["train", "val", "test"]].idxmax(axis=1)
# video_info_split = video_info.assign(split=split)

# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for ax, feat in zip(axes, ["native_speaker", "num_speakers", "background_music"]):
#     (pd.crosstab(video_info_split["split"], video_info_split[feat], normalize="index")
#        .plot(kind="bar", stacked=True, ax=ax))
#     ax.set_title(f"split × {feat}")
#     ax.set_ylabel("proportion")
#     ax.tick_params(axis="x", rotation=0)
# plt.tight_layout()
# plt.show()


## Ego4D exploration and cleaning

In [ ]:
set(av_train.columns) == set(av_val.columns)

In [ ]:
set(av_train.videos[0].keys()) == set(av_val.videos[0].keys())

In [ ]:
{v["split"] for v in av_train.videos}, {v["split"] for v in av_val.videos}

In [ ]:
tr = {v["video_uid"] for v in av_train.videos}
va = {v["video_uid"] for v in av_val.videos}
tr & va

In [ ]:
av = pd.concat([av_train, av_val], ignore_index=True)

In [ ]:
av.head()

In [ ]:
to_interim(config, av, "ego4d", "av", source_keys=["av_train", "av_val"])

In [ ]:
videos = av.videos

In [ ]:
len(videos), videos[0].keys()

In [ ]:
videos[0]["video_uid"], videos[1]["video_uid"]

In [ ]:
videos = pd.DataFrame(av.videos.tolist())
#   .explode("clips", ignore_index=True)
#   .assign(clip_uid=lambda d: d["clips"].str["clip_uid"])
#   .drop(columns="clips"))

In [ ]:
videos.head()

In [ ]:
len(videos)

In [ ]:
len(videos.video_uid.unique())

In [ ]:
videos.isnull().sum()

In [ ]:
videos.isna().sum()

In [ ]:
clips = av_train.videos[0]["clips"]

In [ ]:
len(clips)

In [ ]:
list(clips[0].keys())

In [ ]:
clips[0]["video_uid"], clips[0]["source_clip_uid"], clips[0]["clip_uid"]

In [ ]:
(
    clips[0]["video_start_sec"],
    clips[0]["video_end_sec"],
    clips[0]["clip_start_sec"],
    clips[0]["clip_end_sec"],
)

In [ ]:
(
    clips[0]["video_end_sec"] - clips[0]["video_start_sec"],
    clips[0]["clip_end_sec"] - clips[0]["clip_start_sec"],
)

In [ ]:
(
    clips[0]["video_start_frame"],
    clips[0]["video_end_frame"],
    clips[0]["clip_start_frame"],
    clips[0]["clip_end_frame"],
)

In [ ]:
clips[0]["valid"]

In [ ]:
clips = (
    videos[["split", "clips"]]
    .explode("clips", ignore_index=True)
    .reset_index(drop=True)
    .pipe(
        lambda d: pd.concat(
            [d.drop(columns="clips"), pd.json_normalize(d["clips"])], axis=1
        )
    )
)
# clips = to_table(av_train.videos.tolist(),["clips"],[])

In [ ]:
clips.head()

In [ ]:
len(clips)

In [ ]:
clips.clip_uid.is_unique

In [ ]:
clips.describe()

In [ ]:
clips.info()

In [ ]:
clips.clip_uid.nunique()

In [ ]:
clips.video_uid.nunique()

In [ ]:
clips.source_clip_uid.nunique()

In [ ]:
clips.isna().sum()

In [ ]:
clips.isnull().sum()

In [ ]:
len(clips[clips.split == "val"])

In [ ]:
clips[clips.missing_voice_segments.map(len) > 0]

In [ ]:
clips.valid.describe()

In [ ]:
persons = (
    clips[["clip_uid", "persons"]]
    .explode("persons", ignore_index=True)
    .reset_index(drop=True)
    .pipe(
        lambda d: pd.concat(
            [d.drop(columns="persons"), pd.json_normalize(d["persons"])], axis=1
        )
    )
)

In [ ]:
persons.head()

In [ ]:
persons.info()

In [ ]:
persons.isna().sum()

In [ ]:
persons.isnull().sum()

In [ ]:
persons.describe()

In [ ]:
persons[persons.camera_wearer].describe()

In [ ]:
len(persons.person_id.unique())

In [ ]:
persons["tracking_paths"].map(type).value_counts()

In [ ]:
persons[(~persons.camera_wearer) & (persons.tracking_paths.map(len) == 0)]

In [ ]:
persons[(persons.camera_wearer) & (persons.tracking_paths.map(len) != 0)]

In [ ]:
persons.tracking_paths[1]

In [ ]:
tracking_paths = (
    persons[["clip_uid", "person_id", "tracking_paths"]]
    .explode("tracking_paths", ignore_index=True)
    .dropna(subset=["tracking_paths"])
    .reset_index(drop=True)
    .pipe(
        lambda d: pd.concat(
            [d.drop(columns="tracking_paths"), pd.json_normalize(d["tracking_paths"])],
            axis=1,
        )
    )
)

In [ ]:
tracking_paths.head()

In [ ]:
tracking_paths.describe()

In [ ]:
tracking_paths.info()

In [ ]:
(
    tracking_paths.track_id.is_unique,
    tracking_paths.track_id.nunique(),
    tracking_paths.track_id.isna().sum(),
)

In [ ]:
tracking_paths[tracking_paths.track_id == "track_0"]

In [ ]:
tracking_paths.unmapped_frames_count.describe()

In [ ]:
tracking_paths.loc[tracking_paths.unmapped_frames_count > 0]

In [ ]:
tracking_paths.suspect.describe()

In [ ]:
tracks = (
    tracking_paths[["clip_uid", "person_id", "track_id", "track"]]
    .explode("track", ignore_index=True)
    .reset_index(drop=True)
    .pipe(
        lambda d: pd.concat(
            [d.drop(columns="track"), pd.json_normalize(d["track"])], axis=1
        )
    )
)

In [ ]:
len(tracks)

In [ ]:
tracks.head()

In [ ]:
tracks.isna().sum()

In [ ]:
tracks.video_frame.describe()

In [ ]:
tracks.width.describe()

In [ ]:
tracks.describe()

In [ ]:
voice_segments = (
    persons[["clip_uid", "voice_segments"]]
    .explode("voice_segments", ignore_index=True)
    .reset_index(drop=True)
    .pipe(
        lambda d: pd.concat(
            [d.drop(columns="voice_segments"), pd.json_normalize(d["voice_segments"])],
            axis=1,
        )
    )
)

In [ ]:
voice_segments.head()

In [ ]:
voice_segments.info()

In [ ]:
a = set(voice_segments[voice_segments.person.isna()].clip_uid)

In [ ]:
b = set(clips[clips.split == "val"].clip_uid)

In [ ]:
len(a), len(b)

In [ ]:
a == b

In [ ]:
len(a - b)

In [ ]:
len(b - a)

In [ ]:
nan_mask = voice_segments.person.isna()

In [ ]:
nan_mask

In [ ]:
nan_mask.mean()

In [ ]:
voice_segments[nan_mask]

In [ ]:
voice_segments.loc[nan_mask].notna().sum()

In [ ]:
voice_segments[nan_mask].groupby("clip_uid").size()

In [ ]:
clips.columns

In [ ]:
transcriptions = (
    clips[["clip_uid", "transcriptions"]]
    .explode(column="transcriptions", ignore_index=True)
    .reset_index(drop=True)
    .pipe(
        lambda d: pd.concat(
            [d.drop(columns="transcriptions"), pd.json_normalize(d["transcriptions"])],
            axis=1,
        )
    )
)

In [ ]:
transcriptions.head()

In [ ]:
transcriptions.info()

In [ ]:
transcriptions[transcriptions.transcription.isna()]

In [ ]:
social_segments_talking = (
    clips[["clip_uid", "social_segments_talking"]]
    .explode(column="social_segments_talking", ignore_index=True)
    .reset_index(drop=True)
    .pipe(
        lambda d: pd.concat(
            [
                d.drop(columns="social_segments_talking"),
                pd.json_normalize(d["social_segments_talking"]),
            ],
            axis=1,
        )
    )
)

In [ ]:
social_segments_talking.head()

In [ ]:
social_segments_talking[social_segments_talking.target.isna()].is_at_me.describe()

In [ ]:
len(social_segments_talking[~social_segments_talking.is_at_me])

In [ ]:
social_segments_talking.target.isna().sum()

In [ ]:
len(social_segments_talking[~social_segments_talking.is_at_me])

In [ ]:
social_segments_talking[
    (~social_segments_talking.is_at_me) & (social_segments_talking.target.notna())
]

In [ ]:
social_segments_looking = (
    clips[["clip_uid", "social_segments_looking"]]
    .explode(column="social_segments_looking", ignore_index=True)
    .reset_index(drop=True)
    .pipe(
        lambda d: pd.concat(
            [
                d.drop(columns="social_segments_looking"),
                pd.json_normalize(d["social_segments_looking"]),
            ],
            axis=1,
        )
    )
)

In [ ]:
social_segments_looking.head()

In [ ]:
social_segments_looking.info()

In [ ]:
social_segments_looking.isna().sum()

In [ ]:
social_segments_looking[social_segments_looking.person.isna()]

In [ ]:
social_segments_looking.is_at_me.describe()

In [ ]:
social_segments_looking.person.describe()

In [ ]:
clips.columns

In [ ]:
clips_clean = clips.copy().drop(
    [
        "persons",
        "valid",
        "transcriptions",
        "social_segments_talking",
        "social_segments_looking",
        "missing_voice_segments",
        "camera_wearer.person_id",
        "camera_wearer.camera_wearer",
        "camera_wearer.tracking_paths",
        "camera_wearer.voice_segments",
    ],
    axis=1,
)

In [ ]:
persons_clean = (
    persons.copy()
    .drop(columns=["tracking_paths", "voice_segments"])
    .rename(columns={"camera_wearer": "is_camera_wearer"})
)

In [ ]:
tracking_paths_clean = tracking_paths.copy().drop(columns=["track", "suspect"])

In [ ]:
tracks_clean = (
    tracks.copy().drop(columns=["clip_frame"]).rename(columns={"frame": "clip_frame"})
)
tracks_clean = tracks_clean[(tracks_clean[["width", "height"]] > 0).all(axis=1)]

In [ ]:
voice_segments_clean = (
    voice_segments.copy().dropna().rename(columns={"person": "person_id"})
)

In [ ]:
transcriptions_clean = transcriptions.copy().dropna()

In [ ]:
social_segments_talking_clean = social_segments_talking.copy().dropna()

In [ ]:
social_segments_looking_clean = (
    social_segments_looking.copy().dropna().drop(columns=["target", "is_at_me"])
)

In [ ]:
video_info_clean = (
    video_info.copy()
    .rename(columns={"video_speaker_id": "camera_wearer"})
    .assign(
        split=lambda v: v[["train", "val", "test"]].idxmax(axis=1).astype("category")
    )
    .drop(columns=["train", "val", "test"])
)

In [ ]:
ground_truth_clean = ground_truth.copy().dropna()
ground_truth_clean = ground_truth_clean.groupby(
    ["conversation_id", "speaker_id", "startTime", "endTime"],
    as_index=False,
    sort=False,
).agg(word=("word", "".join))

In [ ]:
video_info.keys()

In [ ]:
video_info_clean.head()

In [ ]:
clips_clean.head()

In [ ]:
persons_clean.head()

In [ ]:
tracking_paths_clean.head()

In [ ]:
transcriptions_clean.head()

In [ ]:
voice_segments_clean.head()

In [ ]:
transcriptions_clean[
    transcriptions_clean.clip_uid == "fc2b2014-9dc4-4a5d-8a1d-25a6911bff7c"
]

In [ ]:
voice_segments_clean[
    voice_segments_clean.clip_uid == "fc2b2014-9dc4-4a5d-8a1d-25a6911bff7c"
]

In [ ]:
len(persons_clean), len(voice_segments_clean)

In [ ]:
vs_keys = set(map(tuple, voice_segments_clean[["clip_uid", "person_id"]].values))
p_keys = set(map(tuple, persons_clean[["clip_uid", "person_id"]].values))

In [ ]:
missing = vs_keys - p_keys
len(missing)

In [ ]:
silent = p_keys - vs_keys
len(silent)

In [ ]:
interim_paths = {
    # Ego4D
    "clips_clean": to_interim(
        config,
        clips_clean,
        "ego4d",
        "clips_clean",
        source_keys="av",
        source_stage="interim",
    ),
    "persons_clean": to_interim(
        config,
        persons_clean,
        "ego4d",
        "persons_clean",
        source_keys="av",
        source_stage="interim",
    ),
    "tracking_paths_clean": to_interim(
        config,
        tracking_paths_clean,
        "ego4d",
        "tracking_paths_clean",
        source_keys="av",
        source_stage="interim",
    ),
    "tracks_clean": to_interim(
        config,
        tracks_clean,
        "ego4d",
        "tracks_clean",
        source_keys="av",
        source_stage="interim",
    ),
    "voice_segments_clean": to_interim(
        config,
        voice_segments_clean,
        "ego4d",
        "voice_segments_clean",
        source_keys="av",
        source_stage="interim",
    ),
    "transcriptions_clean": to_interim(
        config,
        transcriptions_clean,
        "ego4d",
        "transcriptions_clean",
        source_keys="av",
        source_stage="interim",
    ),
    "social_segments_talking_clean": to_interim(
        config,
        social_segments_talking_clean,
        "ego4d",
        "social_segments_talking_clean",
        source_keys="av",
        source_stage="interim",
    ),
    "social_segments_looking_clean": to_interim(
        config,
        social_segments_looking_clean,
        "ego4d",
        "social_segments_looking_clean",
        source_keys="av",
        source_stage="interim",
    ),
    # EgoCom
    "video_info_clean": to_interim(
        config,
        video_info_clean,
        "egocom",
        "video_info_clean",
        source_keys="video_info",
        source_stage="interim",
    ),
    "ground_truth_clean": to_interim(
        config,
        ground_truth_clean,
        "egocom",
        "ground_truth_clean",
        source_keys="ground_truth",
        source_stage="interim",
    ),
}

interim_paths

In [ ]:
import pandas as pd

from conv_wm.config import get_path, load_config

config = load_config()
clips_clean = pd.read_parquet(get_path(config, "ego4d", "interim", "clips_clean"))

print("shape:", clips_clean.shape)
print("\ncolumns:")
print(clips_clean.columns.tolist())

print("\ndtypes:")
print(clips_clean.dtypes)

print("\nnulls:")
print(clips_clean.isna().sum())

print("\nuniqueness:")
print("rows:", len(clips_clean))
print("clip_uid:", clips_clean["clip_uid"].nunique())
print("source_clip_uid:", clips_clean["source_clip_uid"].nunique())

print("\nsplits:")
print(clips_clean["split"].value_counts(dropna=False))